# Dashboard Prep

This notebook is used to prepare the files needed for the Streamlit dashboard. 

In [1]:
from pathlib import Path 
import json
import joblib
import numpy as np
import pandas as pd

In [4]:
DATA_PATH = Path("../data/processed/clean_listings.csv")
MODEL_PATH = Path("../models/best_airbnb_price_model.joblib")

df = pd.read_csv(DATA_PATH)
model = joblib.load(MODEL_PATH)

print("Data shape:", df.shape)
df.head()

Data shape: (37422, 22)


,id,price,log_price,host_is_superhost,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,...,beds,minimum_nights,maximum_nights,availability_365,number_of_reviews,review_scores_rating,review_scores_cleanliness,review_scores_location,review_scores_value,amenities_count
0,2708,67.22,4.222738,1.0,Hollywood,34.09625,-118.34605,Private room in rental unit,Private room,1,...,1.0,30.0,1125.0,301,47,4.87,4.94,4.96,4.87,77
1,2732,213.56,5.368589,0.0,Santa Monica,34.00440,-118.48095,Private room in rental unit,Private room,1,...,1.0,1.0,27.0,350,24,4.41,4.58,4.91,4.22,19
2,6033,99.63,4.611450,0.0,Woodland Hills,34.16887,-118.64478,Entire bungalow,Entire home/apt,3,...,NaN,30.0,1125.0,270,19,4.38,4.00,4.65,4.29,32
3,6931,95.21,4.566533,1.0,Hollywood,34.09626,-118.34372,Private room in rental unit,Private room,1,...,1.0,30.0,1125.0,259,39,4.86,4.92,4.69,4.75,72
4,7874,113.00,4.736198,0.0,Bellflower,33.87687,-118.11444,Private room in home,Private room,2,...,1.0,1.0,730.0,103,26,4.77,4.88,4.77,4.77,22


## Dashboard Options

In [6]:
neighborhood_options = sorted(df["neighbourhood_cleansed"].dropna().unique().tolist())
property_type_options = sorted(df["property_type"].dropna().unique().tolist())
room_type_options = sorted(df["room_type"].dropna().unique().tolist())

print("Number of neighborhoods:", len(neighborhood_options))
print("Number of property types:", len(property_type_options))
print("Room types:", room_type_options)

Number of neighborhoods: 264
Number of property types: 87
Room types: ['Entire home/apt', 'Hotel room', 'Private room', 'Shared room']


## Default Dashboard Inputs

In [8]:
dashboard_defaults = {
    "host_is_superhost": int(df["host_is_superhost"].mode()[0]),
    "neighbourhood_cleansed": df["neighbourhood_cleansed"].mode()[0],
    "latitude": float(df["latitude"].median()),
    "longitude": float(df["longitude"].median()),
    "property_type": df["property_type"].mode()[0],
    "room_type": df["room_type"].mode()[0],
    "accommodates": int(df["accommodates"].median()),
    "bathrooms": float(df["bathrooms"].median()),
    "bedrooms": float(df["bedrooms"].median()),
    "beds": float(df["beds"].median()),
    "minimum_nights": int(df["minimum_nights"].median()),
    "maximum_nights": int(df["maximum_nights"].median()),
    "availability_365": int(df["availability_365"].median()),
    "number_of_reviews": int(df["number_of_reviews"].median()),
    "review_scores_rating": float(df["review_scores_rating"].median()),
    "review_scores_cleanliness": float(df["review_scores_cleanliness"].median()),
    "review_scores_location": float(df["review_scores_location"].median()),
    "review_scores_value": float(df["review_scores_value"].median()),
    "amenities_count": int(df["amenities_count"].median()),
}

dashboard_defaults

{'host_is_superhost': 0,
 'neighbourhood_cleansed': 'Long Beach',
 'latitude': 34.06017887510622,
 'longitude': -118.33822306991634,
 'property_type': 'Entire home',
 'room_type': 'Entire home/apt',
 'accommodates': 4,
 'bathrooms': 1.0,
 'bedrooms': 2.0,
 'beds': 2.0,
 'minimum_nights': 4,
 'maximum_nights': 365,
 'availability_365': 276,
 'number_of_reviews': 8,
 'review_scores_rating': 4.91,
 'review_scores_cleanliness': 4.89,
 'review_scores_location': 4.9,
 'review_scores_value': 4.82,
 'amenities_count': 40}

## Test Dashboard-Style Prediction

In [11]:
sample_input = pd.DataFrame([dashboard_defaults])

log_prediction = model.predict(sample_input)[0]
price_prediction = np.expm1(log_prediction)

print("Predicted nightly price: $", round(price_prediction, 2))

Predicted nightly price: $ 338.98


## Save Dashboard Metadata

In [12]:
APP_DIR = Path("../app")
APP_DIR.mkdir(parents=True, exist_ok=True)

dashboard_metadata = {
    "neighborhood_options": neighborhood_options,
    "property_type_options": property_type_options,
    "room_type_options": room_type_options,
    "defaults": dashboard_defaults,
}

with open(APP_DIR / "dashboard_metadata.json", "w") as f:
    json.dump(dashboard_metadata, f, indent=2)

print("Saved dashboard metadata to:", APP_DIR / "dashboard_metadata.json")

Saved dashboard metadata to: ../app/dashboard_metadata.json


## Market Insight Summaries

In [14]:
REPORTS_DIR = Path("../reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

room_type_summary = (
    df.groupby("room_type")
    .agg(
        avg_price=("price", "mean"),
        median_price=("price", "median"),
        num_listings=("price", "count")
    )
    .sort_values("avg_price", ascending=False)
)

room_type_summary.to_csv(REPORTS_DIR / "room_type_summary.csv")
room_type_summary

,avg_price,median_price,num_listings
room_type,,,
Hotel room,485.707374,316.33,396
Entire home/apt,402.221062,279.59,27983
Private room,132.833477,86.00,8898
Shared room,122.370207,50.88,145


In [15]:
neighborhood_summary = (
    df.groupby("neighbourhood_cleansed")
    .agg(
        avg_price=("price", "mean"),
        median_price=("price", "median"),
        num_listings=("price", "count")
    )
    .sort_values("avg_price", ascending=False)
)

neighborhood_summary.to_csv(REPORTS_DIR / "neighborhood_summary.csv")
neighborhood_summary.head(15)

,avg_price,median_price,num_listings
neighbourhood_cleansed,,,
Beverly Crest,1194.744625,1079.470,160
Unincorporated Catalina Island,1148.625000,1317.750,8
Malibu,1039.390298,858.300,302
Hollywood Hills West,987.646697,853.330,545
Bel-Air,985.002037,841.815,54
Avalon,914.965290,848.000,259
Unincorporated Santa Monica Mountains,833.340807,616.330,161
Palos Verdes Estates,795.443889,374.575,18
Manhattan Beach,735.553049,580.250,410


In [16]:
print("Dashboard prep files created:")
print("Metadata:", (APP_DIR / "dashboard_metadata.json").exists())
print("Room type summary:", (REPORTS_DIR / "room_type_summary.csv").exists())
print("Neighborhood summary:", (REPORTS_DIR / "neighborhood_summary.csv").exists())

Dashboard prep files created:
Metadata: True
Room type summary: True
Neighborhood summary: True
